# Definitions

In [ ]:
%reload_ext autoreload
%autoreload 2
%pylab inline
import numpy as np
from os.path import join, isfile
import sys
from glob import glob
import h5py
sys.path.insert(0, '/home/did/RTC/SMART-G/')
from Voigt_gpu import absorptionCoefficient_Voigt_gpu
from luts.luts import LUT, MLUT, Idx, read_mlut, read_mlut_hdf5, merge
from smartg.atmosphere import AtmAFGL, rod, simps

sys.path.insert(0, '/home/did/RTC/smaccl/')
from smaccl import type_coeff, coeff, get_smac_coeffs
from scipy.interpolate import interp1d

import warnings
warnings.filterwarnings("ignore")

def spherical_alb(L, zenith='Zenith angles'):
    from scipy.integrate import simps
    '''
    Compute albedo over dimension (theta)
    L: reflectance LUT
    theta: name of the zenith axis in degrees
    returns the irradiance value or a LUT for the remainding dimensions
    '''
    mu = (L.axis(zenith, aslut=True)*np.pi/180.).apply(np.cos)
    
    return 2*(mu*L).reduce(simps, zenith, x=-mu[:])


def fitlin(Z, axis, ol="z ~", report=False):
    import pandas
    # For statistics. Requires statsmodels 5.0 or more
    from statsmodels.formula.api import ols
    # Analysis of Variance (ANOVA) on linear models
    from statsmodels.stats.anova import anova_lm
    '''
    Compute the linear model parameters
    Z is the dataset and axis is a list of variables
    returns the parameters estimations
    if report = True, a report on the estimation procedure is given
    '''
    name = 'xyuvw'
    assert ((len(axis) <= 5) & (len(axis) >= 1))
    datadict = {'z': Z.flatten()}
    for k in range(len(axis)) :
        datadict[name[k]] = axis[k].flatten()
        ol +=(' + ' + name[k])
    data = pandas.DataFrame(datadict)
    # Fit the model
    model = ols(ol, data).fit()
    if report:
        print(model.summary())

        print("\nRetrieving manually the parameter estimates:")
        print(model._results.params)

        # Peform analysis of variance on fitted linear model
        anova_results = anova_lm(model)

        print('\nANOVA results')
        print(anova_results)
    
    return model._results.params


# From SMAC
def ref_aer_simple(ksiD, us, uv, wo, gc, taup):
    aer_phase = aer_pha_model(mycoeffs, ksiD, k)
    cksi = np.cos(np.radians(ksiD))
    
    ak2 = (1. - wo)*(3. - wo*3*gc)
    ak  = np.sqrt(ak2)
    e   = -3*us*us*wo /  (4*(1. - ak2*us*us) )
    f   = -(1. - wo)*3*gc*us*us*wo / (4*(1. - ak2*us*us) )
    dp  = e / (3*us) + us*f
    d   = e + f
    b   = 2*ak / (3. - wo*3*gc)
    delta = np.exp( ak*taup )*(1. + b)*(1. + b) - np.exp(-ak*taup)*(1. - b)*(1. - b)
    ww  = wo/4.
    ss  = us / (1. - ak2*us*us)
    q1  = 2. + 3*us + (1. - wo)*3*gc*us*(1. + 2*us)
    q2  = 2. - 3*us - (1. - wo)*3*gc*us*(1. - 2*us)
    q3  = q2*np.exp( -taup/us )
    c1  =  ((ww*ss) / delta) * ( q1*np.exp( ak*taup)*(1. + b) + q3*(1. - b) )
    c2  = -((ww*ss) / delta) * ( q1*np.exp(-ak*taup)*(1. - b) + q3*(1. + b) )
    cp1 =  c1*ak / ( 3. - wo*3*gc )
    cp2 = -c2*ak / ( 3. - wo*3*gc )
    z   = d  - wo*3*gc*uv*dp + wo*aer_phase/4.
    x   = c1 - wo*3*gc*uv*cp1
    y   = c2 - wo*3*gc*uv*cp2
    aa1 = uv / (1. + ak*uv)
    aa2 = uv / (1. - ak*uv)
    aa3 = us*uv / (us + uv) 

    aer_ref = x*aa1* (1. - np.exp( -taup/aa1 ) )
    aer_ref = aer_ref + y*aa2*( 1. - np.exp( -taup / aa2 )  )
    aer_ref = aer_ref + z*aa3*( 1. - np.exp( -taup / aa3 )  )
    aer_ref = aer_ref / ( us*uv )
    
    return aer_ref


def ref_aer_model(mycoeffs, ksiD, us, uv, wo, gc, taup, k):
    aer_ref = ref_aer_simple(ksiD, us, uv, wo, gc, taup)
    xx      = taup * (1/us+1/uv) * cksi
    Res_aer = mycoeffs['Resa1'][k] + mycoeffs['Resa2'][k] *xx +\
              mycoeffs['Resa3'][k] *xx**2 + mycoeffs['Resa4'][k] *xx**3
    
    return aer_ref - Res_aer


def aer_pha_model(mycoeffs, ksiD, k):
    
    return mycoeffs['a0P'][k]+ mycoeffs['a1P'][k]*ksiD +\
                  mycoeffs['a2P'][k]*ksiD**2 + mycoeffs['a3P'][k]*ksiD**3 +\
                  mycoeffs['a4P'][k]*ksiD**4

def ref_ray_simple(cksi, taur, us, uv, Peq):
    ray_phase   = 0.7190443 * (1. + cksi*cksi) + 0.0412742
    ray_ref     = (taur * ray_phase)/(4*us*uv)
    ray_ref     = ray_ref * Peq
    
    return ray_ref
    
def ref_ray_model(mycoeffs, cksi, taur, us, uv, Peq, k):
    ray_ref     = ref_ray_simple(cksi, taur, us, uv, Peq)
    ray_phase   = 0.7190443 * (1. + cksi*cksi) + 0.0412742
    xx          = taur*ray_phase / (us*uv)
    Res_ray     = mycoeffs['Resr1'][k] + mycoeffs['Resr2'][k] *xx + mycoeffs['Resr3'][k] *xx**2
    
    return ray_ref - Res_ray

def ref_atm_model(mycoeffs, cksi, taur, us, uv, Peq, wo, gc, taup, k ):
    taurz   = taur * Peq
    tautot  = taup + taurz
    xx      = tautot * (1/us+1/uv) * cksi
    kdiD    = np.degrees(np.arccos(cksi))
    Res_atm = mycoeffs['Rest1'][k] + mycoeffs['Rest2'][k] *xx + mycoeffs['Rest3'][k] *xx**2 +\
                                     mycoeffs['Rest4'][k] *xx**3
    
    return Res_atm + ref_ray_model(mycoeffs, cksi, taur, us, uv, Peq, k) +\
                     ref_aer_model(mycoeffs, ksiD, us, uv, wo, gc, taup, k)

# SMAC Fits

In [ ]:
'''#------------------------------
#--- Parameters
#------------------------------
lut_dir   = '/rfs/proj/C3S/res/lut_v1/'
gas_dir   = '/rfs/proj/C3S/res/gas_trans_v1/'
# Sensor
sensor    = 'S3B_OLCI'
# artdeco simulation filename
f_artdeco = 'LUT_'+sensor.lower()+'_hitran2012_midres.h5'
# aerosol optical properties filename
f_opt     = 'opt_opac_cont_avg_'+sensor.lower()+'_hitran2012.h5'
# gaseous absorption fit parameters filename
f_gas     = 'smac_coeff_'+sensor.lower()+'.h5'
# aerosols Relative Humidity reference
RH        = 80. # %
# Fit limit on VZA
VZA_LIM   = 50.
# Fit limit on SZA
SZA_LIM   = 70.
# Fit limit on AOT at 550 nm
TAUP550_LIM = 0.6
# Standard surface pressure
P0        = 1013.25 # hPa

#------------------------------
#--- Inputs
#------------------------------
lut_artdeco = read_mlut_hdf5(lut_dir + f_artdeco)
lut_opt     = read_mlut_hdf5(lut_dir + f_opt)
lut_gas     = h5py.File(gas_dir + f_gas, "r")'''


#------------------------------
#--- Inputs
#------------------------------
lut_dir   = '/rfs/proj/C3S/res/lut_v2/'
gas_dir   = '/rfs/proj/C3S/res/gas_trans_v2/'
# Sensor
sensor    = 'S3A_OLCI'
# artdeco simulation filename
f_artdeco = 'LUT_'+sensor.upper()+'_test.h5'

# gaseous absorption fit parameters filename
f_gas     = 'smac_coeff_'+sensor.upper()+'.h5'

lut_artdeco_all = read_mlut_hdf5(lut_dir + f_artdeco)
lut_gas     = h5py.File(gas_dir + f_gas, "r")

In [ ]:
#------------------------------
#--- Parameters
#------------------------------
# aerosols Relative Humidity reference
RH        = 80. # %
# Fit limit on VZA
VZA_LIM   = 50.
# Fit limit on SZA
SZA_LIM   = 70.
# Fit limit on AOT at 550 nm
TAUP550_LIM = 1.
# Standard surface pressure
P0        = 1013.25 # hPa

#------------------------------
#--- aerosol optical properties
#------------------------------

# Source file
f    = h5py.File(lut_dir + f_artdeco)
grp  = f.require_group("aer_model")
opt_file = np.copy(grp["opt_file"]).tostring().decode().rstrip('\x00')
print(opt_file)
print()

# creation of the LUT with all aerosol models
M = []
naer_type = len(grp["aer_type"])
opt_aer_type = {}
for iaer in range(naer_type):
    opt_aer_type[grp["aer_type"][iaer].tostring().decode()] = grp["aer_opt_type"][iaer].tostring().decode()
    m=read_mlut_hdf5(opt_file, group=grp["aer_opt_type"][iaer].tostring().decode())
    m.set_attr('aer_type', iaer)
    M.append(m)
lut_opt = merge(M, ['aer_type'])  

# creation of dictionnaries with model assemblages
# Scale heights 
z_aer_type = {}
for iaer in range( naer_type):
    z_aer_type[grp["aer_type"][iaer].tostring().decode()] = grp["aer_z_type"][iaer]
# relative humidities
rh_aer_model = {}
for iaer in range( naer_type):
    rh_aer_model[grp["aer_type"][iaer].tostring().decode()] = np.copy(grp["model_rh"][iaer,:])
# AOT's fraction
frac_aer_model = {}
for iaer in range( naer_type):
    frac_aer_model[grp["aer_type"][iaer].tostring().decode()] = np.copy(grp["model_aot_fraction"][iaer,:])
f.close()

# Show Models mapping
iaer_lut=0
### Cext/Cext_550
c = ['r','g','y','m','c']
for k,key in enumerate(opt_aer_type.keys()):
    (lut_opt['Cext'].sub()[k, Idx(rh_aer_model[key][iaer_lut]), :]/\
     lut_opt['Cext'].sub()[k, Idx(rh_aer_model[key][iaer_lut]), Idx(0.550)]).plot(fmt='+-'+c[k], 
     label="{:6}<->{:>22}".format(key, opt_aer_type[key]))
legend()
ylim(0.4, 1.5)

### SSA
figure()
for iaer_lut in range(148):
    for k,key in enumerate(opt_aer_type.keys()):
        wo = lut_opt['single_scattering_albedo'].sub()[k, Idx(rh_aer_model[key][iaer_lut]), :]
        wo.plot(fmt='+-'+c[k])
ylim(0.7, 1.0)

In [ ]:
for k,key in enumerate(opt_aer_type.keys()):
    (lut_opt['Cext'].sub()[k, Idx(rh_aer_model[key][iaer_lut]), :]/\
     lut_opt['Cext'].sub()[k, Idx(rh_aer_model[key][iaer_lut]), Idx(0.550)]).plot(fmt='+-'+c[k], 
     label="{:6}<->{:>22}".format(key, opt_aer_type[key]))
legend()
ylim(0.4, 1.5)

In [ ]:
f,ax=subplots(nrows=4)
f.set_size_inches(12,8)
f.set_dpi(100)
width=0.15
c = ['r','g','y','m','c']
N = len(frac_aer_model['sulf'])
x = np.arange(N)+1
N=N//4
for i in range(4):
    for k,key in enumerate(opt_aer_type.keys()):
        y = frac_aer_model[key]
        sca(ax[i])
        bar(x[i*N:(i+1)*N]+k*width, y[i*N:(i+1)*N], width, color=c[k],
            label="{:6}<->{:>22}".format(key, opt_aer_type[key]))
        ylim(0,1)
        grid()
ax[0].legend(bbox_to_anchor=(1,1))

In [ ]:
M = []
for iaer_lut in range(148):
    for k,key in enumerate(opt_aer_type.keys()):
        l = lut_opt['Cext'].sub()[k, Idx(rh_aer_model[key][iaer_lut]), :] * \
            frac_aer_model[key][iaer_lut]/\
            lut_opt['Cext'].sub()[k, Idx(rh_aer_model[key][iaer_lut]), Idx(0.550)]
        if (k==0) : a1taup = l
        else :      a1taup+= l
        a1taup.desc = 'wo'
        m=a1taup.to_mlut()
        m.set_attr('iaer', iaer_lut)
        M.append(m)
    a1taup.plot(fmt='+-')
ylim(0.4, 1.5)
a1taup_lut_ = merge(M,['iaer'])[0]

In [ ]:
M = []
for iaer_lut in range(148):
    for k,key in enumerate(opt_aer_type.keys()):
        l = lut_opt['single_scattering_albedo'].sub()[k, Idx(rh_aer_model[key][iaer_lut]), :] * \
            frac_aer_model[key][iaer_lut]
        if (k==0) : wo = l
        else :      wo+= l
        wo.desc = 'wo'
        m=wo.to_mlut()
        m.set_attr('iaer', iaer_lut)
        M.append(m)
    wo.plot(fmt='+-')
ylim(0.7, 1.0)
wo_lut_ = merge(M,['iaer'])[0]

In [ ]:
_,mu   = np.meshgrid(lut_opt.axis('wavelengths'), lut_opt.axis('mu'))
M = []
for iaer_lut in range(148):
    for k,key in enumerate(opt_aer_type.keys()):
        l = lut_opt['p11_phase_function'].sub()[k, :, Idx(rh_aer_model[key][iaer_lut]), :] * \
            lut_opt['single_scattering_albedo'].sub()[k, Idx(rh_aer_model[key][iaer_lut]), :] * \
            frac_aer_model[key][iaer_lut]
        if (k==0) : P11 = l
        else :      P11+= l

    P11.desc = 'P11'
    g      = 0.5*np.trapz(mu * P11[:, Idx(lut_opt.axis('wavelengths'))], x=lut_opt.axis('mu'), axis=0)
    #P11.sub({'wavelengths':Idx(0.550)}).apply(np.log10).plot(fmt='-')
    plot(lut_opt.axis('wavelengths'), g)
    m=P11.to_mlut()
    m.set_attr('iaer', iaer_lut)
    M.append(m)
#ylim(-1.5,3)
ylim(0.55, 0.8); grid(); xlabel('wavelengths'); ylabel('asymetry parameter')
P11_lut_ = merge(M,['iaer'])[0]

In [ ]:
lut_opt.describe()

In [ ]:
lut_artdeco_all.describe()

In [ ]:
'''# Spectral dependency of aerosols OT
a1Taup = lut_opt['Cext']   [Idx(RH), Idx(wl_1)]/\
         lut_opt['Cext550'][Idx(RH)]
# Aerosols single scattering albedo
wo     = lut_opt['SSA']    [Idx(RH), Idx(wl_1)]
# Aerosols phase function
P11    = lut_opt['P11'].sub()[:, Idx(RH), Idx(wl_1)]
# Aerosols asymetry parameter
_,mu   = np.meshgrid(wl_1, mu_1)
g      = 0.5*np.trapz(mu * P11[:, Idx(wl_1)], x=mu_1, axis=0)'''

iaer_lut = 4


lut_artdeco = lut_artdeco_all.sub({'iaermodel':iaer_lut})
#------------------------------
#--- Some Axes
#------------------------------
# 1-dimensional filtered axes
us_1      = np.cos(np.radians(lut_artdeco.axis('sza')[lut_artdeco.axis('sza')<SZA_LIM]))
uv_1      = np.cos(np.radians(lut_artdeco.axis('vza')[lut_artdeco.axis('vza')<VZA_LIM]))
taup550_1 = lut_artdeco.axis('tauaer')[lut_artdeco.axis('tauaer')<TAUP550_LIM]
Peq_1     = lut_artdeco.axis('Psurf')/P0
raa_1     = np.pi - np.radians(lut_artdeco.axis('raa')) # to convert from artdeco to standard RAA
mu_1      = lut_opt.axis('mu')
wl_1      = lut_artdeco.axis('wvl_c')

_,mu  = np.meshgrid(wl_1, mu_1)
def integ(tab, x=mu, axis=0):
    return 0.5 * np.trapz(x*tab, x=mu_1, axis=axis)

P11_lut     = P11_lut_.sub({    'iaer':iaer_lut}).sub({'wavelengths': Idx(wl_1)})
wo_lut      = wo_lut_.sub( {    'iaer':iaer_lut}).sub({'wavelengths': Idx(wl_1)})
a1taup_lut  = a1taup_lut_.sub( {'iaer':iaer_lut}).sub({'wavelengths': Idx(wl_1)})
g_lut       = P11_lut.reduce(integ, axis=0)
g_lut.desc  = 'g'

# Some plots about aerosol model
fig, ax = subplots()
ax.plot(wl_1, g_lut.data,  '-ok', label='g')
ax.plot(wl_1, wo_lut.data, '-sk', label='ssa')
ax.set_ylim([0.6, 1])
ax.legend()
ax.set_title(sensor+' bands: continental average aerosols')
ax.set_ylabel('g, ssa')
ax.set_xlabel(r'$\lambda (nm)$')
ax2 = ax.twinx()
ax2.plot(wl_1, a1taup_lut.data, '-sr')
ax2.set_ylim([0., 1.6])
ax2.set_ylabel(r'$\tau(\lambda)/\tau(550)$', color='r')
_ = ax2.set_yticklabels([0,0.2,0.4,0.6,0.8,1,1.2,1.4,1.6], color='r')
#fig.savefig('/home/documents/Copernicus/CGLOPS_Lot1/Sentinel-3-EVO/Figures/Aerosols_OLCI.png', dpi=150)
#fig.savefig('/home/documents/Copernicus/CGLOPS_Lot1/Sentinel-3-EVO/Figures/Aerosols_SLSTR.png', dpi=150)

In [ ]:
# SMAC coefficients array
mycoeffs  = np.zeros(wl_1.size, dtype=type_coeff, order='C')

# Loop on bands
for k, wav in enumerate(wl_1):
    mycoeffs['bandname'][k] = sensor+'_band_{:02d}'.format(k+1)
    # taur  = get_tau_rayleigh(wav, P0) # standard ROD for P0=1013.25 hPa
    # standard ROD for P0=1013.25 hPa, 400 ppm CO2, ground level and latitude=45° according to Bodahine 99
    taur  = np.squeeze(rod(wav, array(400.), 45., 0., P0))
    print ('---\n{:.2f}\n---'.format(wav*1e3))
    mycoeffs['taur'][k]   = taur
    
    print('aerosols optical properties')
    '''    P11   = lut_opt['P11'][:, Idx(RH), Idx(wav)]  # aerosols phase function
    gc    = 0.5 * np.trapz(mu_1 * P11, x=mu_1)    # aerosols asymetry parameter
    wo    = lut_opt['SSA'][Idx(RH),  Idx(wav)]    # aerosols single scattering albedo
    a1taup= lut_opt['Cext'][Idx(RH), Idx(wav)]/\
            lut_opt['Cext550'][Idx(RH)]'''
    P11 = P11_lut[:, Idx(wav)]
    gc  = g_lut[k]
    wo  = wo_lut[k]
    a1taup = a1taup_lut[k]
    a0taup= 0.
    mycoeffs['a0taup'][k] = a0taup
    mycoeffs['a1taup'][k] = a1taup
    mycoeffs['gc']    [k] = gc
    mycoeffs['wo']    [k] = wo
    print('a0taup:{:12.5e} a1taup:{:12.5e} gc:{:12.5e} w0:{:12.5e}'.format(a0taup,a1taup,gc,wo))  
    
    print('gaseous transmission')
    for g1 in lut_gas.keys():
        for g2 in lut_gas[g1].keys():          
            print('{:3} {:12.5e} {:12.5e}'.format(g2, lut_gas[g1][g2]['a'][k], 
                                                  lut_gas[g1][g2]['n'][k])  )
            mycoeffs['a'+g2][k] = lut_gas[g1][g2]['a'][k]
            mycoeffs['n'+g2][k] = lut_gas[g1][g2]['n'][k]
    #!!!! ARTDECO ozone in DU, SMAC in cm.atm
    mycoeffs['ao3'] *= 1e3
    #!!!!
    
    #----      
    print('aerosols and Rayleigh transmission')
    # extact 3-dimensional filtered array of total atmospheric total transmission
    tteta_LUT = lut_artdeco['trans_flux_down'].sub({'sza':    Idx(lambda x: x<SZA_LIM)})\
                                              .sub({'tauaer': Idx(lambda x: x<TAUP550_LIM)})\
                                              .sub({'wvl_c':  Idx(wav)})\
                                              .data
    # 3-dimensional filtered meshed axes
    us, taup550, Peq = np.meshgrid(us_1, taup550_1, Peq_1, indexing='ij')
    # fit total transmission
    res = fitlin(tteta_LUT, axis=[taup550/us, Peq/(1.+us), 1./(1.+us)], report=False)
    print('a0T:{:12.5e} a1T:{:12.5e} a2T:{:12.5e} a3T:{:12.5e}'.format(*res))
    mycoeffs['a0T'][k] = res[0]
    mycoeffs['a1T'][k] = res[1]
    mycoeffs['a2T'][k] = res[2]
    mycoeffs['a3T'][k] = res[3]
    #---- 
    
    
    #----
    print('atmosphere spherical albedo')
    alb_LUT = spherical_alb(lut_artdeco['refl_flux_down'], zenith='sza')\
                                .sub({'tauaer': Idx(lambda x: x<TAUP550_LIM)})\
                                .sub({'wvl_c':Idx(wav)})\
                                .data

    taup550, Peq = np.meshgrid(taup550_1, Peq_1, indexing='ij')
    res = fitlin(alb_LUT, axis=[Peq, taup550, taup550*taup550], report=False)
    print('a3s:{:12.5e} a0s:{:12.5e} a1s:{:12.5e} a2s:{:12.5e}'.format(*res))
    mycoeffs['a0s'][k] = res[1]
    mycoeffs['a1s'][k] = res[2]
    mycoeffs['a2s'][k] = res[3]
    mycoeffs['a3s'][k] = res[0]
    #----
    
    #----
    print('Rayleigh reflectance')
    ray_ref_LUT = lut_artdeco['reflectance_toa_ray'].sub({'sza':   Idx(lambda x: x<SZA_LIM)})\
                                                    .sub({'vza':   Idx(lambda x: x<VZA_LIM)})\
                                                    .sub({'bhr':   0})\
                                                    .sub({'wvl_c': Idx(wav)})\
                                                    .data

    us, uv, raa, Peq = np.meshgrid(us_1, uv_1, raa_1, Peq_1, indexing='ij')
    cksi        = - us*uv - np.sqrt(1. - us*us) * np.sqrt (1. - uv*uv) * np.cos(raa)
    cksi2       = np.unique(cksi)
    ksiD        = np.degrees(np.arccos(cksi))
    ksiD2       = np.degrees(np.arccos(cksi2))
    ray_ref     = ref_ray_simple(cksi, taur, us, uv, Peq)
    
    ray_phase   = 0.7190443 * (1. + cksi*cksi) + 0.0412742
    xx          = taur*ray_phase/us/uv
    res = fitlin(ray_ref - ray_ref_LUT, axis=[xx, xx**2], report=False)

    print('Resr1:{:12.5e} Resr2:{:12.5e} Resr3:{:12.5e}'.format(*res))
    mycoeffs['Resr1'][k] = res[0]
    mycoeffs['Resr2'][k] = res[1]
    mycoeffs['Resr3'][k] = res[2]
    #----
    
    #----
    print('aerosols reflectance')
    '''aer_phase_LUT = lut_opt['P11'].sub({'humidity':   Idx(RH)})\
                                  .sub({'wvl_c': Idx(wav)})[Idx(cksi2)]'''
    aer_phase_LUT = P11_lut[Idx(cksi2), Idx(wav)]
    res = fitlin(aer_phase_LUT, axis=[ksiD2, ksiD2**2, ksiD2**3, ksiD2**4], report=False)
    print('a0P:{:12.5e} a1P:{:12.5e} a2P:{:12.5e} a3P:{:12.5e} a4P:{:12.5e}'.format(*res))
    mycoeffs['a0P'][k] = res[0]
    mycoeffs['a1P'][k] = res[1]
    mycoeffs['a2P'][k] = res[2]
    mycoeffs['a3P'][k] = res[3]
    mycoeffs['a4P'][k] = res[4]

    '''plot (ksiD2.flatten() , aer_phase_LUT.flatten(), '.')
    plot (ksiD2.flatten() , aer_pha_model(mycoeffs, ksiD2, k).flatten(), '.r')
    figure()'''
    #----

    #----
    aer_ref_LUT = lut_artdeco['reflectance_toa_aer'].sub({'sza':   Idx(lambda x: x<SZA_LIM)})\
                                                    .sub({'vza':   Idx(lambda x: x<VZA_LIM)})\
                                                    .sub({'tauaer':Idx(lambda x: x<TAUP550_LIM)})\
                                                    .sub({'bhr':   0})\
                                                    .sub({'wvl_c': Idx(wav)})\
                                                    .data
    us, uv, raa, taup550 = np.meshgrid(us_1, uv_1, raa_1, taup550_1, indexing='ij')
    taup        = a0taup + a1taup * taup550 
    cksi        = - us*uv - np.sqrt(1. - us*us) * np.sqrt (1. - uv*uv) * np.cos(raa)
    ksiD        = np.degrees(np.arccos(cksi))

    aer_ref     = ref_aer_simple(ksiD, us, uv, wo, gc, taup)

    xx  = taup*(1/us+1/uv)*cksi
    res = fitlin(aer_ref - aer_ref_LUT,  axis=[xx, xx**2, xx**3], report=False)
    print('Resa1:{:12.5e} Resa2:{:12.5e} Resa3:{:12.5e} Resa4:{:12.5e}'.format(*res))
    mycoeffs['Resa1'][k] = res[0]
    mycoeffs['Resa2'][k] = res[1]
    mycoeffs['Resa3'][k] = res[2]
    mycoeffs['Resa4'][k] = res[3]
    Res_aer     = mycoeffs['Resr1'][k] + mycoeffs['Resr2'][k] *xx + mycoeffs['Resr3'][k] *xx**2
    
    #plot (ksiD.flatten() , (ref_aer_model(mycoeffs, ksiD, us, uv, wo, gc, taup, k) - aer_ref_LUT).flatten(), '.')
    '''figure()
    plot(ksiD.flatten() , ref_aer_simple(ksiD, us, uv, wo, gc, taup).flatten()-\
         aer_ref_LUT.flatten(),'.')'''
    #----

    #----
    print('reflectance de couplage aerosols Rayleigh')
    tot_ref_LUT      = lut_artdeco['reflectance_toa'].sub({'sza':   Idx(lambda x: x<SZA_LIM)})\
                                                     .sub({'vza':   Idx(lambda x: x<VZA_LIM)})\
                                                     .sub({'tauaer':Idx(lambda x: x<TAUP550_LIM)})\
                                                     .sub({'bhr':   0})\
                                                     .sub({'wvl_c': Idx(wav)})\
                                                     .data

    us, uv, raa, taup550, Peq = np.meshgrid(us_1, uv_1, raa_1, taup550_1, Peq_1, indexing='ij')
    taup        = a0taup + a1taup * taup550 
    cksi        = - us*uv - np.sqrt(1. - us*us) * np.sqrt (1. - uv*uv) * np.cos(raa)
    ksiD        = np.degrees(np.arccos(cksi))
    taurz       = taur * Peq
    tautot      = taup + taurz

    ref_aer_m   = ref_aer_model(mycoeffs, ksiD, us, uv, wo, gc, taup, k)
    ref_ray_m   = ref_ray_model(mycoeffs, cksi, taur, us, uv, Peq, k)

    xx  = tautot * (1/us+1/uv) * cksi
    res = fitlin(tot_ref_LUT - (ref_aer_m + ref_ray_m) , axis=[xx, xx**2, xx**3], report=False)
    print('Rest1:{:12.5e} Rest2:{:12.5e} Rest3:{:12.5e} Rest4:{:12.5e}'.format(*res))
    mycoeffs['Rest1'][k] = res[0]
    mycoeffs['Rest2'][k] = res[1]
    mycoeffs['Rest3'][k] = res[2]
    mycoeffs['Rest4'][k] = res[3]
    #----

## Quality check

In [ ]:
# Example of accuracy
# Aerosol reflectance
k=0
aer_ref_LUT = lut_artdeco['reflectance_toa_aer'].sub({'sza':   Idx(lambda x: x<SZA_LIM)})\
                                                .sub({'vza':   Idx(lambda x: x<VZA_LIM)})\
                                                .sub({'tauaer':Idx(lambda x: x<TAUP550_LIM)})\
                                                .sub({'bhr':   0})\
                                                .sub({'wvl_c': k}).describe()
a0taup =    mycoeffs['a0taup'][k]
a1taup =    mycoeffs['a1taup'][k]
us, uv, raa, taup550 = np.meshgrid(us_1, uv_1, raa_1, taup550_1, indexing='ij')
taup        = a0taup + a1taup * taup550 
cksi        = - us*uv - np.sqrt(1. - us*us) * np.sqrt (1. - uv*uv) * np.cos(raa)
ksiD        = np.degrees(np.arccos(cksi))
gc =    mycoeffs['gc']    [k] 
wo =    mycoeffs['wo']    [k]
smac_aer_model = LUT(ref_aer_model(mycoeffs, ksiD, us, uv, wo, gc, taup, k), axes=[aer_ref_LUT.axis('sza'),
                                                                                   aer_ref_LUT.axis('vza'),
                                                                                   aer_ref_LUT.axis('raa'),
                                                                                   aer_ref_LUT.axis('tauaer')],
                                                                             names=['sza','vza','raa','tauaer']).describe()
f=figure()
plot(smac_aer_model[Idx(45), Idx(20), :, :], aer_ref_LUT[Idx(45), Idx(20), :, :], '.')
xlim([0., 0.04])
ylim([0., 0.04])
grid()
plot([0., 0.04],[0., 0.04],'k--')
xlabel('SMAC')
ylabel('ARTDECO')
title('TOA aer. reflectance; SZA=45, VZA=20, band: {}_{:02d}'.format(sensor,k+1))
#leg=('{:.3f},'*13).format(*aer_ref_LUT.axis('tauaer'))
#legend(leg.split(sep=','), title=r'$\tau_a (550)$', ncol=2)
#f.savefig('/home/documents/Copernicus/CGLOPS_Lot1/Sentinel-3-EVO/Figures/Fit_accuracy_example.png', dpi=150)

In [ ]:
# Example of accuracy
# Total reflectance

ref_LUT = lut_artdeco['reflectance_toa'].sub({'sza':   Idx(lambda x: x<SZA_LIM)})\
                                                .sub({'vza':   Idx(lambda x: x<VZA_LIM)})\
                                                .sub({'tauaer':Idx(lambda x: x<TAUP550_LIM)})\
                                                .sub({'bhr':   0})\
                                                .sub({'wvl_c': k}).describe()
a0taup =    mycoeffs['a0taup'][k]
a1taup =    mycoeffs['a1taup'][k]
taur   =    mycoeffs['taur'][k]
us, uv, raa, taup550, Peq = np.meshgrid(us_1, uv_1, raa_1, taup550_1, Peq_1, indexing='ij')
taup        = a0taup + a1taup * taup550 
cksi        = - us*uv - np.sqrt(1. - us*us) * np.sqrt (1. - uv*uv) * np.cos(raa)
ksiD        = np.degrees(np.arccos(cksi))
gc =    mycoeffs['gc']    [k] 
wo =    mycoeffs['wo']    [k]
smac_atm_model = LUT(ref_atm_model(mycoeffs, cksi, taur, us, uv, Peq, wo, gc, taup, k ), axes=[ref_LUT.axis('sza'),
                                                                                   ref_LUT.axis('vza'),
                                                                                   ref_LUT.axis('raa'),
                                                                                   ref_LUT.axis('tauaer'),
                                                                                   ref_LUT.axis('Psurf')],
                                                                             names=['sza','vza','raa','tauaer','Psurf']).describe()

f=figure()
plot(smac_atm_model[Idx(45), Idx(20), :, :, Idx(950.)], ref_LUT[Idx(45), Idx(20), :, :, Idx(950.)], '.')
xlim([0.1, 0.2])
ylim([0.1, 0.2])
grid()
plot([0.1, 0.2],[0.1, 0.2],'k--')
xlabel('SMAC')
ylabel('ARTDECO')
title('TOA reflectance; SZA=45, VZA=20, band: {}_{:02d}'.format(sensor,k+1))
#leg=('{:.3f},'*13).format(*aer_ref_LUT.axis('tauaer'))
#legend(leg.split(sep=','), title=r'$\tau_a (550)$', ncol=2)
#f.savefig('/home/documents/Copernicus/CGLOPS_Lot1/Sentinel-3-EVO/Figures/Fit_accuracy_example_TOA.png', dpi=150)

In [ ]:
sys.path.insert(0, '/home/did/RTC/smac/')
from smac import smac_inv, smac_dir

In [ ]:
#example
theta_s = 40.
theta_v = 0.
phi_s   = 220.
phi_v   = 60.
r_surf  = 0.05
uo3=0.3
taua = 0.0
uh2o = 0.3
######################################lecture des coefs_smac
coefs    = coeff('/home/did/RTC/smacg/COEFFS/coef_AATSR_550_CONT.dat')
r_toa2 = smac_dir(r_surf,  theta_s, phi_s, theta_v, phi_v,1013,taua,uo3,uh2o, coefs)
r_surf = smac_inv(r_toa2 , theta_s, phi_s, theta_v, phi_v,1013,taua,uo3,uh2o, coefs)

print(r_surf, r_toa2)

In [ ]:
coefs2   = coeff('/home/did/RTC/smacg/COEFFS/coef_AATSR_550_CONT.dat')
###################################### nouveaux coefs_smac
for (field,v), c in zip(type_coeff, mycoeffs[0]):
    coefs2.__dict__.update({field:c})

r_toa2 = smac_dir(r_surf,  theta_s, phi_s, theta_v, phi_v,1013,taua,uo3,uh2o, coefs2)
r_surf = smac_inv(r_toa2 , theta_s, phi_s, theta_v, phi_v,1013,taua,uo3,uh2o, coefs2)

print(r_surf, r_toa2)

In [ ]:
for key in coefs.__dict__.keys():
    print('{:6s} {:13.5e} {:13.5e}'.format(key, coefs.__dict__[key], coefs2.__dict__[key]))

In [ ]:
np.save('/home/did/RTC/smacg/COEFFS/'+sensor.lower()+'_hitran2012_opac_cont_avg_v1.0', mycoeffs)

# SRFs

In [ ]:
# Plot All sensors SRFS
from c3slib import SRF
for sensor in SRF().split(sep=', '):
    xLimits, wLimits, fwhm, central_wvl, odr, srf_wvl, srf = SRF(sensor)
    for w,s in zip(srf_wvl, srf):
        plot(w,s)

## Gaseous Transmission

In [ ]:
sensor = 'S2A_MSI'
xLimits, wLimits, fwhm, central_wvl, odr, srf_wvl, srf = SRF(sensor)

In [ ]:
# Download Hitran lines ?
FETCH = False
# get HAPI 
sys.path.insert(0, '/home/did/RTC/HITRAN/')
import hapi
warnings.filterwarnings("ignore")
molecules     = ["CO2", "O2", "H2O", "CH4", "NO2", "CO", "N2O"]
molecules2    = ["O3", "CO2", "O2", "H2O", "CH4", "NO2", "CO", "N2O"]
hitran_dir    = '/home/did/RTC/smaccl/COEFFS/data_hitran/{}'.format(sensor)

col           = ['C'+str(i) for i in range(len(molecules2))]

# Initialize Local spectroscopic database
hapi.db_begin(hitran_dir)

for mol in molecules:
    ids=[]
    for k in hapi.ISO_ID.keys():
        if hapi.ISO_ID[k][-1] in mol: ids.append(k)
    for b,lim in enumerate(xLimits):
        numin=lim[0]; numax=lim[1]
        database = sensor+'_band_{:02d}_database_{}'.format(b+1,mol)
        try:
            if FETCH : hapi.fetch_by_ids(database, ids, numin=numin, numax=numax)
        except:
            print("NO lines\n-----------\n")

In [ ]:
NL   = 11 # number of levels
znew = np.linspace(100.,0.,num=NL)
# define the atmosphere for absorption coefficient computation, keep ozone
atm  = AtmAFGL('afglus', grid=znew, O3=0.)
# Set spectral interval in wavelength space
dl   = .01 # resolution (nm)

for k,lim in enumerate(xLimits):
    lmin = 1e7/lim[1]
    lmax = 1e7/lim[0]
    NW   = int((lmax-lmin)/dl)+1
    # convert into the wavenumber space for Hitran
    vmin = lim[0]
    vmax = lim[1]
    dv   = (vmax-vmin)/NW

    fig, ax = subplots()
    ax.plot(srf_wvl[k][:], srf[k][:], 'k')
    
    for j,mol in enumerate(molecules2):
        if mol != 'O3':
            database = sensor+'_band_{:02d}_database_{}'.format(k+1,mol)
            fdatabase= hitran_dir+'/'+database+'.header'
            coef_    = []
            if isfile(fdatabase) :
                # loop on each vertical level
                coef_    = []
                for p,t,dens,z in zip(atm.prof.P,atm.prof.T,atm.prof.__dict__['dens_'+mol.lower()],atm.prof.z):
                    nu,coef   = absorptionCoefficient_Voigt_gpu(SourceTables=database,
                                HITRAN_units=True,
                                OmegaRange=[vmin,vmax], OmegaStep=dv, GammaL='gamma_self',
                                Environment={'p':p/1013.,'T':t})

                coef_.append(array(coef))
                cc = np.vstack(coef_).T
                dd = atm.prof.__dict__['dens_'+mol.lower()]
                ab =  cc *  dd * 1e5
                #convert the increasing wavenumber grid (cm-1) into an increasing wavelength grid (in nm)
                wl = 1e7/nu[::-1]
                ab = ab[::-1]
                trans = np.exp(-np.trapz(ab, x=-atm.prof.z, axis=-1))
            else:
                wl = np.linspace(lmin, lmax, num=NW)
                trans = np.ones_like(wl)
           
        else:
            wl = np.linspace(lmin, lmax, num=NW)
            pro  = AtmAFGL('afglus', grid=znew).calc(wl, phase=False)
            trans = np.exp(-pro['OD_g'][:,-1])

        srf_int = interp1d(srf_wvl[k][:], srf[k][:], fill_value='extrapolate')(wl)

        trans_int = np.trapz(srf_int * trans, x=wl)/np.trapz(srf_int, x=wl)
        
        if (trans_int<0.999):
            print('{} {:02d} {:3s} {:6.4f}'.format(sensor, k+1,  mol.lower(), trans_int))
            ax.plot(wl, trans, label=mol, color=col[j])
    ax.set_xlabel(r'$\lambda (nm)$')
    ax.set_ylabel('$T$')
    ax.set_title('%s : band %i'%(sensor,k+1))
    ax.legend()
    fig.savefig('/rfs/proj/C3S/SRFs/'+sensor+'_band{0:02d}_trans.png'.format(k+1), dpi=150)